In [0]:
# ===================================================
# BLOCK 1 — CONFIGURATION (PYTHON)
# ===================================================

from pyspark.sql import functions as F

CATALOG = "semiconplus_portfolio"
LANDING = f"{CATALOG}.simulation.simulated_hourly_equipment_operations_landing"
BRONZE = f"{CATALOG}.bronze.simulated_hourly_equipment_operations"
SILVER = f"{CATALOG}.silver.simulated_hourly_equipment_operations"

spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Simulated hourly OEE publication configuration loaded.")

In [0]:
# ===================================================
# BLOCK 2 — PUBLISH BRONZE (PYTHON)
# ===================================================

bronze_df = (
    spark.table(LANDING)
    .withColumn("_source_table", F.lit(LANDING))
    .withColumn("_source_record_id", F.col("operation_record_id"))
    .withColumn("_ingested_at_utc", F.current_timestamp())
    .withColumn("_pipeline_run_id", F.lit("DAY4_OEE_V1"))
)

(bronze_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(BRONZE))
print(f"Bronze simulated OEE rows: {spark.table(BRONZE).count():,}")

In [0]:
# ===================================================
# BLOCK 3 — PUBLISH VALIDATED SILVER (PYTHON)
# ===================================================

silver_df = (
    spark.table(BRONZE)
    .select(
        F.col("operation_record_id").cast("string"),
        F.col("operation_hour_utc").cast("timestamp"),
        F.col("operation_hour_local").cast("timestamp"),
        F.col("production_date").cast("date"),
        F.col("production_hour_index").cast("int"),
        F.col("site_id").cast("string"), F.col("equipment_id").cast("string"),
        F.col("scheduled_time_seconds").cast("long"),
        F.col("approved_planned_downtime_seconds").cast("long"),
        F.col("planned_production_time_seconds").cast("long"),
        F.col("unplanned_downtime_seconds").cast("long"),
        F.col("operating_time_seconds").cast("long"),
        F.col("short_stop_seconds").cast("long"), F.col("setup_seconds").cast("long"),
        F.col("run_seconds").cast("long"), F.col("total_units").cast("long"),
        F.col("good_units").cast("long"), F.col("rated_units_per_hour").cast("int"),
        F.col("theoretical_output_units").cast("double"),
        F.col("simulation_seed").cast("long"), F.col("simulation_version").cast("string"),
        F.col("simulated_record_flag").cast("boolean"), F.col("record_origin").cast("string"),
        "_source_table", "_source_record_id", "_ingested_at_utc",
        F.current_timestamp().alias("_silver_processed_at_utc"),
    )
)

invalid = silver_df.filter(
    F.col("operation_record_id").isNull() | F.col("operation_hour_utc").isNull()
    | F.col("production_date").isNull() | F.col("equipment_id").isNull()
    | (F.col("planned_production_time_seconds") < 0)
    | (F.col("operating_time_seconds") < 0)
    | (F.col("operating_time_seconds") > F.col("planned_production_time_seconds"))
    | (F.col("good_units") > F.col("total_units"))
    | (F.col("total_units") > F.col("theoretical_output_units") + 1e-9)
    | (~F.col("simulated_record_flag"))
).count()
assert invalid == 0, f"Invalid Silver OEE rows: {invalid}"

(silver_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(SILVER))
print(f"Silver simulated OEE rows: {spark.table(SILVER).count():,}")

In [0]:
# ===================================================
# BLOCK 4 — LAYER RECONCILIATION (PYTHON)
# ===================================================

counts = [("LANDING", spark.table(LANDING).count()),
          ("BRONZE", spark.table(BRONZE).count()),
          ("SILVER", spark.table(SILVER).count())]
display(spark.createDataFrame(counts, ["dataset", "row_count"]))
assert len({x[1] for x in counts}) == 1
assert counts[0][1] == 1051776
print("Simulated hourly OEE layer reconciliation passed.")
